# FRTB SA GIRR — End-to-End Pipeline

Demonstrates delta, vega, and curvature capital charges under CRR3 Art. 325bd/bf/e using **real QRE instruments** and the **FRTB_Sensitivity_Engine**.

| Capital charge | Regulatory article | Calculator |
|---|---|---|
| GIRR delta | CRR3 Art. 325bd/bf | SA_GIRR_Calculator |
| GIRR vega  | CRR3 Art. 325bd/bf | SA_GIRR_Vega_Calculator |
| GIRR curvature | CRR3 Art. 325e/ef | SA_GIRR_Curvature_Calculator |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql

from quant_risk import Bond, IRSwap, VanillaOption, ArrayCurve

from banking_risk.frtb.portfolio import (
    FRTB_Risk_Class,
    Trading_Instrument,
    Standard_Trading_Portfolio,
)
from banking_risk.frtb.sensitivity_engine import FRTB_Sensitivity_Engine
from banking_risk.frtb.girr.delta import SA_GIRR_Calculator
from banking_risk.frtb.girr.vega import SA_GIRR_Vega_Calculator
from banking_risk.frtb.girr.curvature import SA_GIRR_Curvature_Calculator
from banking_risk.frtb.vertex_mapping import (
    FRTB_GIRR_VERTICES,
    FRTB_GIRR_LABELS,
    GIRR_VEGA_VERTICES,
)
from banking_risk.utils.reporting import Dark_Style

Dark_Style().apply()
p = Dark_Style().palette

## 1. Curve Construction

Build an EUR zero curve and inspect the 10 prescribed FRTB GIRR vertices (CRR3 Art. 325bd).

In [ ]:
maturities = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0])
zero_rates = np.array([0.032, 0.033, 0.034, 0.035, 0.036, 0.037, 0.038, 0.039, 0.040, 0.040, 0.039])

curve = ArrayCurve(maturities, zero_rates)

vertex_rates = [curve.zero_rate(v) for v in FRTB_GIRR_VERTICES]
pd.DataFrame({
    'vertex': FRTB_GIRR_LABELS,
    'years': FRTB_GIRR_VERTICES,
    'zero_rate': vertex_rates,
}).set_index('vertex')

## 2. Portfolio

Mix of EUR and USD GIRR instruments: bonds, swaps, and a swaption (for vega/curvature).

In [ ]:
ql.Settings.instance().evaluationDate = ql.Date(1, 6, 2026)

eur_bond = Bond(
    isin='BUND_5Y', face_value=5_000_000.0, coupon_rate=0.025,
    issue_date='2023-06-01', maturity_date='2028-06-01', currency='EUR',
)
eur_irs = IRSwap(
    notional=10_000_000.0, maturity_years=10, fixed_rate=0.038,
    valuation_date='2026-06-01', pay_fixed=True, currency='EUR',
)

usd_irs = IRSwap(
    notional=5_000_000.0, maturity_years=3, fixed_rate=0.054,
    valuation_date='2026-06-01', pay_fixed=False, currency='USD',
)

eur_swaption = VanillaOption(
    spot=0.038, strike=0.038, expiry_date=ql.Date(1, 6, 2027),
    valuation_date=ql.Date(1, 6, 2026), sigma=0.18,
    option_type='call', notional_=5_000_000.0, currency_='EUR',
    underlying='5Y_IRS',
)

instruments = [
    Trading_Instrument('BUND_5Y', 'EUR', frozenset({FRTB_Risk_Class.GIRR}), eur_bond),
    Trading_Instrument('IRS_10Y_pay', 'EUR', frozenset({FRTB_Risk_Class.GIRR}), eur_irs),
    Trading_Instrument('IRS_3Y_rcv', 'USD', frozenset({FRTB_Risk_Class.GIRR}), usd_irs),
    Trading_Instrument('Swaption_1Y5Y', 'EUR', frozenset({FRTB_Risk_Class.GIRR}), eur_swaption, is_linear=False),
]
portfolio = Standard_Trading_Portfolio(instruments)

pd.DataFrame([
    {'name': ti.name, 'ccy': ti.currency, 'risk_class': list(ti.risk_classes)[0].value, 'linear': ti.is_linear}
    for ti in portfolio.instruments()
])

## 3. GIRR Delta

PV01 at 10 FRTB vertices per currency.

In [ ]:
engine = FRTB_Sensitivity_Engine(portfolio, curve)
delta_sens = engine.girr_delta()

print("Raw delta sensitivities (currency per 1bp):")
for ccy, arr in delta_sens.items():
    print(f"\n{ccy}:")
    df = pd.DataFrame({'vertex': FRTB_GIRR_LABELS, 'DV01': arr})
    print(df[df['DV01'] != 0].to_string(index=False))

In [ ]:
delta_result = SA_GIRR_Calculator().compute(delta_sens)

print(f"GIRR Delta Capital: {delta_result.capital:,.2f}")
display(delta_result.to_table().style.format('{:,.2f}'))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(FRTB_GIRR_LABELS))
w = 0.35

for i, (ccy, arr) in enumerate(delta_sens.items()):
    ax.bar(x + i*w, arr, w, label=ccy, alpha=0.8)

ax.axhline(0, color='white', linewidth=0.8, alpha=0.4)
ax.set_xlabel('Vertex')
ax.set_ylabel('PV01 (currency per 1bp)')
ax.set_title('GIRR Delta Sensitivities')
ax.set_xticks(x + w/2)
ax.set_xticklabels(FRTB_GIRR_LABELS, rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 4. GIRR Vega

Sensitivity to implied volatility changes. Non-linear instruments only.

In [ ]:
vega_sens = engine.girr_vega()

if vega_sens:
    print("Vega sensitivities found.")
    n_vega = len(GIRR_VEGA_VERTICES)
    vega_grids = {ccy: arr.reshape(n_vega, n_vega) for ccy, arr in vega_sens.items()}
    
    for ccy, grid in vega_grids.items():
        print(f"\n{ccy} vega grid (expiry × tenor):")
        df = pd.DataFrame(
            grid,
            index=pd.Index(GIRR_VEGA_VERTICES, name='expiry'),
            columns=pd.Index(GIRR_VEGA_VERTICES, name='tenor'),
        )
        display(df.round(1).replace(0, "-"))
else:
    print("No vega (all instruments linear).")

## 5. GIRR Curvature

Non-linear P&L under full repricing at ±RW bumps. Non-linear instruments only.

In [ ]:
cvr_up, cvr_dn = engine.girr_curvature()

if cvr_up or cvr_dn:
    print("Curvature sensitivities:")
    for ccy in set(list(cvr_up.keys()) + list(cvr_dn.keys())):
        print(f"\n{ccy}:")
        print(f"  CVR_up  : {cvr_up.get(ccy, 0.0):>12,.2f}")
        print(f"  CVR_down: {cvr_dn.get(ccy, 0.0):>12,.2f}")
else:
    print("No curvature (all instruments linear).")

In [ ]:
if cvr_up or cvr_dn:
    curv_result = SA_GIRR_Curvature_Calculator().compute(cvr_up, cvr_dn)
    
    print(f"GIRR Curvature Capital: {curv_result.capital:,.2f}")
    display(curv_result.to_table().style.format('{:,.2f}'))
else:
    print("No curvature capital.")
    curv_result = None

## 6. Combined GIRR Capital

CRR3 Art. 325bb: GIRR capital = delta + vega + curvature.

In [ ]:
components = {
    'GIRR delta': delta_result.capital,
    'GIRR vega': vega_result.capital if vega_result else 0.0,
    'GIRR curvature': curv_result.capital if curv_result else 0.0,
}
total_girr = sum(components.values())
components['Total GIRR'] = total_girr

summary = pd.DataFrame(
    {'capital': components},
)
display(summary.style.format('{:,.2f}'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = [k for k in components.keys() if k != 'Total GIRR']
values = [components[k] for k in labels]
colors = [p['cyan'], p['green'], p['amber']]

ax.bar(labels, values, color=colors, alpha=0.85)
ax.set_ylabel('Capital charge')
ax.set_title(f'GIRR SA Capital Components (Total: {total_girr:,.2f})')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()